### 导入库

In [1]:
import pandas as pd
import numpy as np

### 读入数据

In [2]:
# 从上到下样本逐渐增多
Australia = pd.read_csv("./statlog australian credit approval 690行/australian.dat", sep='\s+',header=None)
German = pd.read_csv("./German/german.data", sep='\s+',header=None)
Taiwan = pd.read_csv('./Default of Credit Card Clients Dataset （台湾）30000行/UCI_Credit_Card.csv')
Platform = pd.read_csv('./预测用户贷款是否违约/train.csv')
# Bank_bill = pd.read_csv('./银行征信数据/bill_detail_train.txt',sep=',',header=None)

In [3]:
# Bank_detail = pd.read_csv('./银行征信数据/bank_detail_train.txt',sep=',',header=None)
# Bank_browse = pd.read_csv('./银行征信数据/browse_history_train.txt',sep=',',header=None)
# Bank_tag = pd.read_csv('./银行征信数据/overdue_train.txt',sep=',',header=None)
# Bank_user = pd.read_csv('./银行征信数据/user_info_train.txt',sep=',',header=None)
# Bank_loan = pd.read_csv('./银行征信数据/loan_time_train.txt',sep=',',header=None)

#### 基本数据统计

##### 是否平衡

In [4]:
print(Australia[14].value_counts())
Australia_major = Australia[14].value_counts()[0]
German.iloc[:,-1] = [label - 1 for label in German.iloc[:,-1]]
print(German[20].value_counts())
German_major = German[20].value_counts()[0]
print(Taiwan['default.payment.next.month'].value_counts())
Taiwan_major = Taiwan['default.payment.next.month'].value_counts()[0]
print(Platform['isDefault'].value_counts())
Platform_major = Platform['isDefault'].value_counts()[0]

14
0    383
1    307
Name: count, dtype: int64
20
0    700
1    300
Name: count, dtype: int64
default.payment.next.month
0    23364
1     6636
Name: count, dtype: int64
isDefault
0    640390
1    159610
Name: count, dtype: int64


##### 是否有缺失值

In [5]:
print(Australia.isnull().sum())

0     0
1     0
2     0
3     0
4     0
5     0
6     0
7     0
8     0
9     0
10    0
11    0
12    0
13    0
14    0
dtype: int64


In [6]:
print(German.isnull().sum())

0     0
1     0
2     0
3     0
4     0
5     0
6     0
7     0
8     0
9     0
10    0
11    0
12    0
13    0
14    0
15    0
16    0
17    0
18    0
19    0
20    0
dtype: int64


In [7]:
print(Taiwan.isnull().sum())

ID                            0
LIMIT_BAL                     0
SEX                           0
EDUCATION                     0
MARRIAGE                      0
AGE                           0
PAY_0                         0
PAY_2                         0
PAY_3                         0
PAY_4                         0
PAY_5                         0
PAY_6                         0
BILL_AMT1                     0
BILL_AMT2                     0
BILL_AMT3                     0
BILL_AMT4                     0
BILL_AMT5                     0
BILL_AMT6                     0
PAY_AMT1                      0
PAY_AMT2                      0
PAY_AMT3                      0
PAY_AMT4                      0
PAY_AMT5                      0
PAY_AMT6                      0
default.payment.next.month    0
dtype: int64


In [8]:
print(Platform.isnull().sum())

id                        0
loanAmnt                  0
term                      0
interestRate              0
installment               0
grade                     0
subGrade                  0
employmentTitle           1
employmentLength      46799
homeOwnership             0
annualIncome              0
verificationStatus        0
issueDate                 0
isDefault                 0
purpose                   0
postCode                  1
regionCode                0
dti                     239
delinquency_2years        0
ficoRangeLow              0
ficoRangeHigh             0
openAcc                   0
pubRec                    0
pubRecBankruptcies      405
revolBal                  0
revolUtil               531
totalAcc                  0
initialListStatus         0
applicationType           0
earliesCreditLine         0
title                     1
policyCode                0
n0                    40270
n1                    40270
n2                    40270
n3                  

### 数据格式统一

#### 分开特征和标签

In [9]:
Australia_X = Australia.iloc[:, :14]
Australia_y = Australia.iloc[:, 14]
German_X = German.iloc[:,:-1]
German_y = German.iloc[:,-1]
Taiwan_X = Taiwan.iloc[:,:-1]
Taiwan_y = Taiwan.iloc[:,-1]
Platform_X = Platform.drop('isDefault', axis=1)
Platform_y = Platform['isDefault']

In [10]:
del Taiwan_X['ID']
del Platform_X['id']
del Platform_X['regionCode']

#### 对于序数属性的处理/二元标称属性

##### German

第一列支票账户余额区间转换为数字，其中没有支票账户A14和A12合并

In [11]:
German_X[0]= German_X[0].str[-1:].astype(int)
German_X[0] = German_X[0].replace(4,2)
German_X[0].value_counts() 

0
2    663
1    274
3     63
Name: count, dtype: int64

第三列信用历史转换为数字

In [12]:
German_X[2]= German_X[2].str[-1:].astype(int)
German_X[2].value_counts() 

2
2    530
4    293
3     88
1     49
0     40
Name: count, dtype: int64

第四列贷款用途，由于类别太多，直接使用独热编码会使得数据很稀疏，因此A40-44归为家具，A45为维修，A46-A48为教育，A49为生意，A50为其他

In [13]:
German_X[3]= German_X[3].str[-2:].astype(int)
German_X[3] = German_X[3].replace({40:0,41: 0, 42: 0,43: 0,44: 0,45: 1,46:2,47:2,48:2,49:3,10:4})
German_X[3].value_counts() 

3
0    810
3     97
2     59
1     22
4     12
Name: count, dtype: int64

第六列存款，将unknown设置为0，因为没有记录绝大部分是没有存款

In [14]:
German_X[5]= German_X[5].str[-1:].astype(int)
German_X[5] = German_X[5].replace(5,0)
German_X[5].value_counts() 

5
1    603
0    183
2    103
3     63
4     48
Name: count, dtype: int64

第七列目前工作从几年前开始，将unemployed设置为1

In [15]:
German_X[6]= German_X[6].str[-1:].astype(int)
German_X[6].value_counts() 

6
3    339
5    253
4    174
2    172
1     62
Name: count, dtype: int64

第十二列，资产种类隐含资产多少的关系

In [16]:
German_X[11]= German_X[11].str[-1:].astype(int)
German_X[11].value_counts() 

11
3    332
1    282
2    232
4    154
Name: count, dtype: int64

第十七列，就业情况，隐含偿还能力

In [17]:
German_X[16]= German_X[16].str[-1:].astype(int)
German_X[16].value_counts() 

16
3    630
2    200
4    148
1     22
Name: count, dtype: int64

第十九列和二十列，二元分类变量

In [18]:
German_X[18]= German_X[18].str[-1:].astype(int)
German_X[18].value_counts() 
German_X[19]= German_X[19].str[-1:].astype(int)
German_X[19].value_counts() 

19
1    963
2     37
Name: count, dtype: int64

##### Platform

subGrade等级变成连续型变量

In [19]:
Platform_X = Platform_X.drop(['grade'],axis=1)

In [20]:
Platform_X.columns

Index(['loanAmnt', 'term', 'interestRate', 'installment', 'subGrade',
       'employmentTitle', 'employmentLength', 'homeOwnership', 'annualIncome',
       'verificationStatus', 'issueDate', 'purpose', 'postCode', 'dti',
       'delinquency_2years', 'ficoRangeLow', 'ficoRangeHigh', 'openAcc',
       'pubRec', 'pubRecBankruptcies', 'revolBal', 'revolUtil', 'totalAcc',
       'initialListStatus', 'applicationType', 'earliesCreditLine', 'title',
       'policyCode', 'n0', 'n1', 'n2', 'n3', 'n4', 'n5', 'n6', 'n7', 'n8',
       'n9', 'n10', 'n11', 'n12', 'n13', 'n14'],
      dtype='object')

In [21]:
Platform_X['subGrade'] = Platform_X['subGrade'].str[0].apply(lambda x: ord(x) - ord('A'))*5+Platform_X['subGrade'].str[1].astype(int)

In [22]:
Platform_X['subGrade'].value_counts()

subGrade
11    50763
9     49516
10    48965
8     48600
12    47068
13    44751
14    44272
7     44227
6     42382
15    40264
5     38045
4     30928
16    30538
17    26528
1     25909
18    23410
3     22655
2     22124
19    21139
20    17838
21    14064
22    12746
23    10925
24     9273
25     8653
26     5925
27     4340
28     3577
29     2859
30     2352
31     1759
32     1231
33      978
34      751
35      645
Name: count, dtype: int64

In [23]:
Platform_X['employmentLength']=Platform_X['employmentLength'].astype(str).replace({'10+ years':20,'< 1 year':0.5,'1 year':1,'2 years':2,'3 years':3,'4 years':4,'5 years':5,'6 years':6,'7 years':7,'8 years':8,'9 years':9}).astype(float)

In [24]:
Platform_X['employmentLength'].value_counts()

employmentLength
20.0    262753
2.0      72358
0.5      64237
3.0      64152
1.0      52489
5.0      50102
4.0      47985
6.0      37254
8.0      36192
7.0      35407
9.0      30272
Name: count, dtype: int64

发生时间，其实可以结合是否同一用户上一次已违约来生成新一列属性，但是我现在直接把他删了，因为对模型优秀程度比较似乎没有什么帮助

In [25]:
Platform_X = Platform_X.drop(['issueDate'],axis=1)

最早信用时间，直接使用年份

In [26]:
Platform_X['earliesCreditLine']=Platform_X['earliesCreditLine'].str[4:].astype(int)

In [27]:
Platform_X.dtypes

loanAmnt              float64
term                    int64
interestRate          float64
installment           float64
subGrade                int64
employmentTitle       float64
employmentLength      float64
homeOwnership           int64
annualIncome          float64
verificationStatus      int64
purpose                 int64
postCode              float64
dti                   float64
delinquency_2years    float64
ficoRangeLow          float64
ficoRangeHigh         float64
openAcc               float64
pubRec                float64
pubRecBankruptcies    float64
revolBal              float64
revolUtil             float64
totalAcc              float64
initialListStatus       int64
applicationType         int64
earliesCreditLine       int32
title                 float64
policyCode            float64
n0                    float64
n1                    float64
n2                    float64
n3                    float64
n4                    float64
n5                    float64
n6        

#### 对于标称属性的处理-把一个标称属性分成多栏

##### German

第九列性别及是否单身，分成性别一栏+单身一栏

In [28]:
#性别,0为女性，1为男性
German_X[20] =German_X[8].apply(lambda x: 1 if x == 'A91' or x == 'A93' or x == 'A94' else 0)
German_X[20].value_counts()

20
1    690
0    310
Name: count, dtype: int64

In [29]:
#单身为0，非单身为1
German_X[21] =German_X[8].apply(lambda x: 0 if x == 'A93' or x == 'A95' else 1)
German_X[21].value_counts()

21
0    548
1    452
Name: count, dtype: int64

第十列，是否有债权人/担保人

In [30]:
German_X.columns

Index([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19,
       20, 21],
      dtype='int64')

In [31]:
dummies = pd.get_dummies(German_X[9],dummy_na=False)
German_X = pd.concat([German_X, dummies], axis=1)
German_X.columns

Index([     0,      1,      2,      3,      4,      5,      6,      7,      8,
            9,     10,     11,     12,     13,     14,     15,     16,     17,
           18,     19,     20,     21, 'A101', 'A102', 'A103'],
      dtype='object')

第十四列，是否有其他分期付款计划

In [32]:
dummies = pd.get_dummies(German_X[13],dummy_na=False)
German_X = pd.concat([German_X, dummies], axis=1)
German_X.columns

Index([     0,      1,      2,      3,      4,      5,      6,      7,      8,
            9,     10,     11,     12,     13,     14,     15,     16,     17,
           18,     19,     20,     21, 'A101', 'A102', 'A103', 'A141', 'A142',
       'A143'],
      dtype='object')

第十五列，住房类型

In [33]:
dummies = pd.get_dummies(German_X[14],dummy_na=False)
German_X = pd.concat([German_X, dummies], axis=1)
German_X.columns

Index([     0,      1,      2,      3,      4,      5,      6,      7,      8,
            9,     10,     11,     12,     13,     14,     15,     16,     17,
           18,     19,     20,     21, 'A101', 'A102', 'A103', 'A141', 'A142',
       'A143', 'A151', 'A152', 'A153'],
      dtype='object')

In [34]:
German_X = German_X.drop(German_X.columns[[8, 9, 13, 14]], axis=1)
German_X.columns

Index([     0,      1,      2,      3,      4,      5,      6,      7,     10,
           11,     12,     15,     16,     17,     18,     19,     20,     21,
       'A101', 'A102', 'A103', 'A141', 'A142', 'A143', 'A151', 'A152', 'A153'],
      dtype='object')

In [35]:
German_X.columns = range(German_X.shape[1])

In [36]:
German_X.columns

RangeIndex(start=0, stop=27, step=1)

#### 对于缺失值的处理

缺失值都不是标签，可以进行填充或者标记，而不直接删除数据；把分类太多的删除

In [37]:
Platform_X['employmentTitle']=Platform_X['employmentTitle'].fillna(Platform_X['employmentTitle'].mean())

Platform_X['employmentLength']=Platform_X['employmentLength'].fillna(Platform_X['employmentLength'].mean())

# Platform_X['postCode']=Platform_X['postCode'].fillna(Platform_X['postCode'].mode()[0])
Platform_X.drop(['postCode'], axis=1, inplace=True)
Platform_X.drop(['title'], axis=1, inplace=True)

Platform_X['dti']=Platform_X['dti'].fillna(Platform_X['dti'].mean())

Platform_X['pubRecBankruptcies']=Platform_X['pubRecBankruptcies'].fillna(Platform_X['pubRecBankruptcies'].mean())

Platform_X['revolUtil']=Platform_X['revolUtil'].fillna(Platform_X['revolUtil'].mean())

# Platform_X['title']=Platform_X['title'].fillna(Platform_X['title'].mode()[0])

Platform_X['n0']=Platform_X['n0'].fillna(Platform_X['n0'].mean())
Platform_X['n1']=Platform_X['n1'].fillna(Platform_X['n1'].mean())
Platform_X['n2']=Platform_X['n2'].fillna(Platform_X['n2'].mean())
Platform_X['n3']=Platform_X['n3'].fillna(Platform_X['n3'].mean())
Platform_X['n4']=Platform_X['n4'].fillna(Platform_X['n4'].mean())
Platform_X['n5']=Platform_X['n5'].fillna(Platform_X['n5'].mean())
Platform_X['n6']=Platform_X['n6'].fillna(Platform_X['n6'].mean())
Platform_X['n7']=Platform_X['n7'].fillna(Platform_X['n7'].mean())
Platform_X['n8']=Platform_X['n8'].fillna(Platform_X['n8'].mean())
Platform_X['n9']=Platform_X['n9'].fillna(Platform_X['n9'].mean())
Platform_X['n10']=Platform_X['n10'].fillna(Platform_X['n10'].mean())
Platform_X['n11']=Platform_X['n11'].fillna(Platform_X['n11'].mean())
Platform_X['n12']=Platform_X['n12'].fillna(Platform_X['n12'].mean())
Platform_X['n13']=Platform_X['n13'].fillna(Platform_X['n13'].mean())
Platform_X['n14']=Platform_X['n14'].fillna(Platform_X['n14'].mean())

In [38]:
Platform_X.isnull().sum()

loanAmnt              0
term                  0
interestRate          0
installment           0
subGrade              0
employmentTitle       0
employmentLength      0
homeOwnership         0
annualIncome          0
verificationStatus    0
purpose               0
dti                   0
delinquency_2years    0
ficoRangeLow          0
ficoRangeHigh         0
openAcc               0
pubRec                0
pubRecBankruptcies    0
revolBal              0
revolUtil             0
totalAcc              0
initialListStatus     0
applicationType       0
earliesCreditLine     0
policyCode            0
n0                    0
n1                    0
n2                    0
n3                    0
n4                    0
n5                    0
n6                    0
n7                    0
n8                    0
n9                    0
n10                   0
n11                   0
n12                   0
n13                   0
n14                   0
dtype: int64

#### 不平衡的处理

过采样比欠采样效果更佳 https://ieeexplore.ieee.org/abstract/document/9078901

新过采样方法 https://www.mdpi.com/2073-8994/13/2/194 

##### SMOTE

imblearn 提供的 SMOTENC 支持混合类型的数据，数值特征使用插值，分类特征随机采样。

In [39]:
from sklearn.neighbors import NearestNeighbors
 
def SMOTE(X, y, N, k=5):
    """
    合成少数类过采样技术（SMOTE）
    参数：
        X (numpy数组): 包含数据点的特征矩阵。
        y (numpy数组): 对应的标签数组（多数类为0，少数类为1）。
        N (int): 生成的合成样本数量。
        k (int, 可选): 考虑的最近邻居数量，默认为5。
    返回：
        X_synthetic (numpy数组): 包含生成样本的合成特征矩阵。
        y_synthetic (numpy数组): 合成样本对应的标签数组。
    """
 
    # 分离多数类和少数类样本
    X_majority = X[y == 0]
    X_minority = X[y == 1]
 
    # 计算每个少数类样本需要生成的合成样本数量
    N_per_sample = N // len(X_minority)
 
    # 如果k大于少数样本数量，则将其减少到可能的最大值
    k = min(k, len(X_minority) - 1)
 
    # 初始化列表以存储合成样本和相应的标签
    synthetic_samples = []
    synthetic_labels = []
 
    # 在少数类样本上拟合k近邻
    knn = NearestNeighbors(n_neighbors=k)
    knn.fit(X_minority)
 
    for minority_sample in X_minority:
        # 查找当前少数类样本的k个最近邻居
        _, indices = knn.kneighbors(minority_sample.reshape(1, -1), n_neighbors=k)
 
        # 随机选择k个邻居并创建合成样本
        for _ in range(N_per_sample):
            # 选出邻居
            neighbor_index = np.random.choice(indices[0])
            neighbor = X_minority[neighbor_index]
 
            # 计算当前少数类样本和邻居之间的差异
            difference = neighbor - minority_sample
 
            # 生成一个0到1之间的随机数
            alpha = np.random.random()
 
            # 创建一个合成样本作为少数类样本和邻居的线性组合
            synthetic_sample = minority_sample + alpha * difference
 
            # 将合成样本及其标签追加到列表中
            synthetic_samples.append(synthetic_sample)
            synthetic_labels.append(1) 
 
    # 将列表转换为numpy数组
    X_synthetic = np.array(synthetic_samples)
    y_synthetic = np.array(synthetic_labels)
 
    # 将原始多数类样本与合成样本合并
    X_balanced = np.concatenate((X_majority, X_synthetic), axis=0)
    y_balanced = np.concatenate((np.zeros(len(X_majority)), y_synthetic), axis=0)
 
    return X_balanced, y_balanced

Australia

In [40]:
from imblearn.over_sampling import SMOTENC
#确定分类特征
categorical_features = [0,2,3,4,5,7,8,9,10,11,13]

# 初始化 SMOTENC
smote_nc = SMOTENC(categorical_features=categorical_features, random_state=2)
# 执行过采样
Australia_X_res, Australia_y_res = smote_nc.fit_resample(Australia_X, Australia_y)

# 转换为 DataFrame 查看结果
Australia_X_res = pd.DataFrame(Australia_X_res, columns=Australia_X.columns)
# Australia_y_res = pd.DataFrame(Australia_y_res)

In [41]:
Australia_y_res.value_counts()

14
0    383
1    383
Name: count, dtype: int64

German

In [42]:
German_X.iloc[0]

0         1
1         6
2         4
3         0
4      1169
5         0
6         5
7         4
8         4
9         1
10       67
11        2
12        3
13        1
14        2
15        1
16        1
17        0
18     True
19    False
20    False
21    False
22    False
23     True
24    False
25     True
26    False
Name: 0, dtype: object

In [43]:
#确定分类特征
categorical_features = [0,3,5,9,11,13,14,15,16,17,18,19,20,21,22,23,24,25,26]

# 初始化 SMOTENC
smote_nc = SMOTENC(categorical_features=categorical_features, random_state=2)
# 执行过采样
German_X_res, German_y_res = smote_nc.fit_resample(German_X, German_y)

# 转换为 DataFrame 查看结果
German_X_res = pd.DataFrame(German_X_res, columns=German_X.columns)
# German_y_res = pd.DataFrame(German_y_res)

In [44]:
German_y_res.value_counts()

20
0    700
1    700
Name: count, dtype: int64

Taiwan

In [45]:
Taiwan_X

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6
0,20000.0,2,2,1,24,2,2,-1,-1,-2,...,689.0,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0
1,120000.0,2,2,2,26,-1,2,0,0,0,...,2682.0,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0
2,90000.0,2,2,2,34,0,0,0,0,0,...,13559.0,14331.0,14948.0,15549.0,1518.0,1500.0,1000.0,1000.0,1000.0,5000.0
3,50000.0,2,2,1,37,0,0,0,0,0,...,49291.0,28314.0,28959.0,29547.0,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0
4,50000.0,1,2,1,57,-1,0,-1,0,0,...,35835.0,20940.0,19146.0,19131.0,2000.0,36681.0,10000.0,9000.0,689.0,679.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,220000.0,1,3,1,39,0,0,0,0,0,...,208365.0,88004.0,31237.0,15980.0,8500.0,20000.0,5003.0,3047.0,5000.0,1000.0
29996,150000.0,1,3,2,43,-1,-1,-1,-1,0,...,3502.0,8979.0,5190.0,0.0,1837.0,3526.0,8998.0,129.0,0.0,0.0
29997,30000.0,1,2,2,37,4,3,2,-1,0,...,2758.0,20878.0,20582.0,19357.0,0.0,0.0,22000.0,4200.0,2000.0,3100.0
29998,80000.0,1,3,1,41,1,-1,0,0,0,...,76304.0,52774.0,11855.0,48944.0,85900.0,3409.0,1178.0,1926.0,52964.0,1804.0


In [46]:
#确定分类特征
categorical_features = ['SEX','MARRIAGE']

# 初始化 SMOTENC
smote_nc = SMOTENC(categorical_features=categorical_features, random_state=2)
# 执行过采样
Taiwan_X_res, Taiwan_y_res = smote_nc.fit_resample(Taiwan_X, Taiwan_y)

# 转换为 DataFrame 查看结果
Taiwan_X_res = pd.DataFrame(Taiwan_X_res, columns=Taiwan_X.columns)
# Taiwan_y_res = pd.DataFrame(Taiwan_y_res)

In [47]:
Taiwan_y_res.value_counts()

default.payment.next.month
1    23364
0    23364
Name: count, dtype: int64

Platform

In [48]:
# #确定分类特征
# categorical_features = ['homeOwnership','verificationStatus','purpose','initialListStatus','policyCode']

# # 初始化 SMOTENC
# smote_nc = SMOTENC(categorical_features=categorical_features, random_state=2)
# # 执行过采样
# Platform_X_res, Platform_y_res = smote_nc.fit_resample(Platform_X, Platform_y)

# # 转换为 DataFrame 查看结果
# Platform_X_res = pd.DataFrame(Platform_X_res, columns=Platform_X.columns)
# # Platform_y_res = pd.DataFrame(Platform_y_res)

In [52]:
# Platform_X_res.to_csv('Platform_X_res.csv',index = False)
Platform_X_res = pd.read_csv('Platform_X_res.csv')
Platform_y_res = pd.read_csv('Platform_y_res.csv')

### Baseline

#### 逻辑回归

In [53]:
# 导入必要的库
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'average_precision']

def logistic_regression(X, y, cv=5, max_iter=5000,scoring = scoring,random_state=42):
    

    # 创建逻辑回归模型
    model = LogisticRegression(max_iter=max_iter,random_state=random_state)

    results = cross_validate(model, X, y, cv=cv, scoring=scoring)
    print(f"平均准确率: {results['test_accuracy'].mean():.2f}")
    print(f"平均精确率: {results['test_precision'].mean():.2f}")
    print(f"平均召回率: {results['test_recall'].mean():.2f}")
    print(f"平均F1得分: {results['test_f1'].mean():.2f}")
    print(f"平均AUC-ROC: {results['test_roc_auc'].mean():.2f}")
    print(f"平均AUC-PR: {results['test_average_precision'].mean():.2f}")
    # # 训练模型
    # model.fit(X_train, y_train)

    # # 测试模型
    # y_pred = model.predict(X_test)

    # # 评估模型
    # accuracy = accuracy_score(y_test, y_pred)
    # conf_matrix = confusion_matrix(y_test, y_pred)
    # report = classification_report(y_test, y_pred)

    # # 打印结果
    # print("模型准确率:", accuracy)
    # print("\n混淆矩阵:\n", conf_matrix)
    # print("\n分类报告:\n", report)


##### Australia

In [54]:
logistic_regression(Australia_X,Australia_y,max_iter=9000)

平均准确率: 0.86
平均精确率: 0.84
平均召回率: 0.86
平均F1得分: 0.85
平均AUC-ROC: 0.93
平均AUC-PR: 0.92


In [55]:
logistic_regression(Australia_X_res,Australia_y_res,max_iter=9000)

平均准确率: 0.87
平均精确率: 0.86
平均召回率: 0.90
平均F1得分: 0.88
平均AUC-ROC: 0.94
平均AUC-PR: 0.94


##### German

In [56]:
logistic_regression(German_X,German_y,max_iter=9000)

平均准确率: 0.74
平均精确率: 0.62
平均召回率: 0.38
平均F1得分: 0.47
平均AUC-ROC: 0.74
平均AUC-PR: 0.56


In [57]:
logistic_regression(German_X_res,German_y_res,max_iter=9000)

平均准确率: 0.76
平均精确率: 0.78
平均召回率: 0.67
平均F1得分: 0.68
平均AUC-ROC: 0.87
平均AUC-PR: 0.86


##### Taiwan

In [58]:
logistic_regression(Taiwan_X,Taiwan_y,max_iter=9000)

D:\anaconda3\lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
D:\anaconda3\lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_resu

平均准确率: 0.81
平均精确率: 0.70
平均召回率: 0.25
平均F1得分: 0.37
平均AUC-ROC: 0.72
平均AUC-PR: 0.50


D:\anaconda3\lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [59]:
logistic_regression(Taiwan_X_res,Taiwan_y_res,max_iter=9000)

D:\anaconda3\lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
D:\anaconda3\lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_resu

平均准确率: 0.67
平均精确率: 0.68
平均召回率: 0.66
平均F1得分: 0.67
平均AUC-ROC: 0.74
平均AUC-PR: 0.74


D:\anaconda3\lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


##### Platform

In [60]:
logistic_regression(Platform_X,Platform_y,max_iter=9000)

D:\anaconda3\lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
D:\anaconda3\lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_resu

平均准确率: 0.80
平均精确率: 0.53
平均召回率: 0.07
平均F1得分: 0.13
平均AUC-ROC: 0.71
平均AUC-PR: 0.37


In [61]:
logistic_regression(Platform_X_res,Platform_y_res,max_iter=9000)

D:\anaconda3\lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
D:\anaconda3\lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_resu

平均准确率: 0.67
平均精确率: 0.67
平均召回率: 0.68
平均F1得分: 0.67
平均AUC-ROC: 0.73
平均AUC-PR: 0.69


#### 决策树

In [62]:
# 导入必要的库
from sklearn.tree import DecisionTreeClassifier

# # 构建决策树分类器
# model = DecisionTreeClassifier(random_state=42)

# # 使用训练集数据训练模型
# model.fit(X_train, y_train)

# # 使用测试集数据进行预测
# y_pred = model.predict(X_test)

# # 评估模型
# accuracy = accuracy_score(y_test, y_pred)
# conf_matrix = confusion_matrix(y_test, y_pred)
# report = classification_report(y_test, y_pred)

# print("模型准确率:", accuracy)
# print("\n混淆矩阵:\n", conf_matrix)
# print("\n分类报告:\n", report)

def descision_tree(X, y,cv=5,scoring = scoring,random_state=42,max_depth=5):

    # 创建逻辑回归模型
    model = DecisionTreeClassifier(random_state=random_state,max_depth = max_depth)

    results = cross_validate(model, X, y, cv=cv, scoring=scoring)
    print(f"平均准确率: {results['test_accuracy'].mean():.2f}")
    print(f"平均精确率: {results['test_precision'].mean():.2f}")
    print(f"平均召回率: {results['test_recall'].mean():.2f}")
    print(f"平均F1得分: {results['test_f1'].mean():.2f}")
    print(f"平均AUC-ROC: {results['test_roc_auc'].mean():.2f}")
    print(f"平均AUC-PR: {results['test_average_precision'].mean():.2f}")

##### Australia

In [63]:
descision_tree(Australia_X,Australia_y,max_depth=20)

平均准确率: 0.81
平均精确率: 0.78
平均召回率: 0.78
平均F1得分: 0.78
平均AUC-ROC: 0.80
平均AUC-PR: 0.71


In [67]:
descision_tree(Australia_X_res,Australia_y_res,max_depth=20)

平均准确率: 0.84
平均精确率: 0.84
平均召回率: 0.86
平均F1得分: 0.85
平均AUC-ROC: 0.84
平均AUC-PR: 0.79


##### German

In [68]:
descision_tree(Taiwan_X,Taiwan_y,max_depth=50)

平均准确率: 0.73
平均精确率: 0.39
平均召回率: 0.42
平均F1得分: 0.40
平均AUC-ROC: 0.62
平均AUC-PR: 0.29


In [69]:
descision_tree(Taiwan_X_res,Taiwan_y_res,max_depth=50)

平均准确率: 0.75
平均精确率: 0.74
平均召回率: 0.77
平均F1得分: 0.75
平均AUC-ROC: 0.75
平均AUC-PR: 0.69


##### Taiwan

In [64]:
descision_tree(Taiwan_X,Taiwan_y,max_depth=50)

平均准确率: 0.73
平均精确率: 0.39
平均召回率: 0.42
平均F1得分: 0.40
平均AUC-ROC: 0.62
平均AUC-PR: 0.29


In [70]:
descision_tree(Taiwan_X_res,Taiwan_y_res,max_depth=50)

平均准确率: 0.75
平均精确率: 0.74
平均召回率: 0.77
平均F1得分: 0.75
平均AUC-ROC: 0.75
平均AUC-PR: 0.69


##### Platform

In [71]:
descision_tree(Platform_X,Platform_y,max_depth=50)

平均准确率: 0.70
平均精确率: 0.28
平均召回率: 0.30
平均F1得分: 0.29
平均AUC-ROC: 0.55
平均AUC-PR: 0.22


In [72]:
descision_tree(Platform_X_res,Platform_y_res,max_depth=50)

平均准确率: 0.78
平均精确率: 0.76
平均召回率: 0.77
平均F1得分: 0.73
平均AUC-ROC: 0.78
平均AUC-PR: 0.72


#### Xgboost

In [72]:
# 导入必要的库
from xgboost import XGBClassifier

# # 假设你已经有了 `Australia_X` 和 `Australia_y`
# X = Australia_X  # 特征数据 (100个样本，2个特征)
# y = Australia_y  # 标签数据 (分类标签)

# # 分割数据集为训练集和测试集 (80% 训练集，20% 测试集)
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# # 构建XGBoost分类器
# model = XGBClassifier(eval_metric='logloss')

# # 使用训练集数据训练模型
# model.fit(X_train, y_train)

# # 使用测试集数据进行预测
# y_pred = model.predict(X_test)

# # 评估模型
# accuracy = accuracy_score(y_test, y_pred)
# conf_matrix = confusion_matrix(y_test, y_pred)
# report = classification_report(y_test, y_pred)

# print("模型准确率:", accuracy)
# print("\n混淆矩阵:\n", conf_matrix)
# print("\n分类报告:\n", report)

def xgboost(X, y,cv=5,scoring = scoring,random_state=2,n_estimators=1000):

    # 创建逻辑回归模型
    model = XGBClassifier(eval_metric='logloss',random_state = random_state,n_estimators=n_estimators)

    results = cross_validate(model, X, y, cv=cv, scoring=scoring)
    print(f"平均准确率: {results['test_accuracy'].mean():.2f}")
    print(f"平均精确率: {results['test_precision'].mean():.2f}")
    print(f"平均召回率: {results['test_recall'].mean():.2f}")
    print(f"平均F1得分: {results['test_f1'].mean():.2f}")
    print(f"平均AUC-ROC: {results['test_roc_auc'].mean():.2f}")
    print(f"平均AUC-PR: {results['test_average_precision'].mean():.2f}")

In [75]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split

def xgboost(X, y,cv=5,scoring = scoring,random_state=2,n_estimators=1000):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    # 创建逻辑回归模型
    model = XGBClassifier(eval_metric='logloss',random_state = random_state,n_estimators=n_estimators)

    # 训练模型
    model.fit(X_train, y_train)
    # 预测
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]  # 取概率值，用于 AUC 和 Average Precision

    # 计算多个指标
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba)
    average_precision = average_precision_score(y_test, y_proba)

    # 打印结果
    print(f"Accuracy: {accuracy:.3f}")
    print(f"Precision: {precision:.3f}")
    print(f"Recall: {recall:.3f}")
    print(f"F1 Score: {f1:.3f}")
    print(f"ROC AUC: {roc_auc:.3f}")
    print(f"Average Precision: {average_precision:.3f}")

##### Australia

In [76]:
xgboost(Australia_X,Australia_y,n_estimators=1000)

Accuracy: 0.877
Precision: 0.840
Recall: 0.824
F1 Score: 0.832
ROC AUC: 0.906
Average Precision: 0.846


In [77]:
xgboost(Australia_X_res,Australia_y_res,n_estimators=1000)

Accuracy: 0.890
Precision: 0.895
Recall: 0.883
F1 Score: 0.889
ROC AUC: 0.955
Average Precision: 0.963


##### German

In [78]:
xgboost(German_X,German_y,n_estimators=1000)

Accuracy: 0.780
Precision: 0.667
Recall: 0.508
F1 Score: 0.577
ROC AUC: 0.803
Average Precision: 0.663


In [79]:
xgboost(German_X_res,German_y_res,n_estimators=1000)

Accuracy: 0.811
Precision: 0.779
Recall: 0.832
F1 Score: 0.804
ROC AUC: 0.902
Average Precision: 0.898


##### Taiwan

In [84]:
xgboost(Taiwan_X,Taiwan_y,n_estimators=1000)

Accuracy: 0.802
Precision: 0.573
Recall: 0.375
F1 Score: 0.454
ROC AUC: 0.752
Average Precision: 0.501


In [81]:
xgboost(Taiwan_X_res,Taiwan_y_res,n_estimators=1000)

Accuracy: 0.840
Precision: 0.855
Recall: 0.820
F1 Score: 0.837
ROC AUC: 0.920
Average Precision: 0.932


##### Platform

In [82]:
xgboost(Platform_X,Platform_y,n_estimators=1000)

Accuracy: 0.802
Precision: 0.496
Recall: 0.143
F1 Score: 0.222
ROC AUC: 0.709
Average Precision: 0.371


In [83]:
xgboost(Platform_X_res,Platform_y_res,n_estimators=1000)

Accuracy: 0.875
Precision: 0.956
Recall: 0.786
F1 Score: 0.863
ROC AUC: 0.927
Average Precision: 0.947


### 隐私性

各方为了保证自己数据的隐私不被泄露，常常不会选择直接把原始数据直接寄送进行统一处理，而是选择在本地进行一定的处理后（例如先进行逻辑回归的500次迭代），然后将结果再进行进一步的集中训练，得到最终的结果。这种方法称为VLR（Vertical Logistic Regression），是联邦学习范式实现的一种方式。

同态加密可以进一步对数据进行秘密处理。使用Paillier加密处理线性部分（如全连接层），对非线性部分（激活函数）在解密后明文计算。（个人觉得实际意义不大，如果线性和非线性的掺杂在一起，会拆分的很稀碎）

### 公平性

计划损失函数加入惩罚项进行衡量